# CruxRNN

**Sequence Similarity, Practice Scheduling, and Interference in a Minimal Recurrent Neural Network**

This notebook walks through **saved, verified pilot simulations**, using the small CSVs and example metadata tracked in the repository. It does not train models. Pilot outputs verify the pipeline and are not final scientific findings. You need basic Python, NumPy, and pandas. The numerical experiments used Python 3.8.10 and PyTorch 1.13.1 CPU.

We compare practicing three movement patterns in separate blocks with practicing the same examples in a shuffled order. Climbing offers an analogy: you might repeat one move pattern or alternate several. The data here are artificial action labels, not climbing measurements. The results concern this network and do not prove anything about human brains.

Read `../RUN_REPORT.md` for completion counts, runtime, and deviations. An earlier final experiment was already executed before the later pilot approval gate; that history is disclosed in `../reports/prior_execution.md`. The new proposed confirmation remains unexecuted and requires approval.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# This supports launching Jupyter from the project or notebooks folder.
root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
assert (root / 'pyproject.toml').exists()
analysis = root / 'results' / 'pilot'
verification = json.loads((analysis / 'verification.json').read_text())
print('Verified pilot run pairs:', len(verification['completed']))
print('Missing pilot run pairs:', len(verification['missing']))


Verified pilot run pairs: 20
Missing pilot run pairs: 0


## A, B, and C

Each pattern is a list of seven actions. Internally the actions are 0, 1, 2, and 3. A family contains three bases, called A, B, and C. The family is regenerated independently for each simulation and distance.

**Hamming distance** counts positions where two lists differ. At k changed positions, B and C each use an action different from A and from each other. Thus A–B, A–C, and B–C all have the same distance k. At all remaining positions the actions agree.


In [2]:
example = json.loads((analysis / 'example.json').read_text())
manifest = example['manifest']
for name in ['A', 'B', 'C']:
    print(name, manifest['main_sequences'][name]['actions_0_based'])
print('Changed positions (Python starts at 0):', manifest['changed_positions_0_based'])

def hamming(first, second):
    return sum(a != b for a, b in zip(first, second))

A = manifest['main_sequences']['A']['actions_0_based']
B = manifest['main_sequences']['B']['actions_0_based']
C = manifest['main_sequences']['C']['actions_0_based']
print('Pairwise distances:', hamming(A, B), hamming(A, C), hamming(B, C))


A [2, 0, 2, 1, 2, 0, 0]
B [2, 3, 2, 1, 2, 0, 1]
C [2, 2, 2, 1, 2, 0, 3]
Changed positions (Python starts at 0): [1, 6]
Pairwise distances: 2 2 2


## Input noise and unchanged targets

The input cue uses **base + 1**, so values are 1–4 before noise. We add Gaussian input noise with standard deviation 0.1. The correct target remains the original action class 0–3; noise never changes the answer.

Development and final evaluation use newly generated noise, with separate deterministic seeds. They contain the same three bases as training but fresh cues. Generalization uses different, unseen bases. The controlled generator also prevents exact overlap with pretraining and D.


In [3]:
example_cue = np.array(example['training_example_cue'])
example_target = np.array(example['training_example_target'])
print('Training cue shape:', example['train_cue_shape'])
print('Pilot held-out noisy cue shape:', example['final_probe_cue_shape'])
print('Clean cue:', example_target + 1)
print('Noisy cue:', np.round(example_cue, 3))
print('Target:', example_target)


Training cue shape: [15000, 1, 7]
Pilot held-out noisy cue shape: [300, 1, 7]
Clean cue: [3 1 3 2 3 1 1]
Noisy cue: [3.005 0.863 3.209 2.136 3.027 1.069 1.103]
Target: [2 0 2 1 2 0 0]


## Model shapes and the complete cue

The cue tensor is `(batch, 1, 7)`: 20 examples in a training batch, one initial time step, and seven features. The **entire intended action pattern is supplied at the beginning**. This task asks the model to decode and unfold a complete cue, not to guess a missing future action.

The RNN has seven hidden units and tanh activation. A linear layer gives four raw logits for the first action. Another linear layer maps these logits back to seven input features, which the recurrent layer uses for the next action. Hidden state resets to zero for each example. Seven action outputs make `(batch, 7, 4)`. Argmax chooses the predicted action. The source also computes one final feedback transition whose result is unused; this timing is preserved.

Cross-entropy measures how much probability the model assigns to the correct targets. **An optimizer update** computes a loss on 20 examples and adjusts weights with SGD. With 15,000 examples, there are 750 main updates. There is one pass through the data, not 750 full passes.


## Practice order and pairing

Blocked training presents all 5000 A examples, then all 5000 B examples, then all 5000 C examples, or another prespecified permutation of the three blocks. Interleaved training shuffles all 15,000 example IDs. Both schedules use exactly the same noisy cues and targets.

For each simulation, one pretrained network is cloned into both schedules, with identical weight hashes and fresh identical SGD optimizers. This **paired design** helps isolate order. The pilot uses five independent IDs at each k. The proposed final confirmation would use 30 IDs per k, with each of the six block orders appearing five times. Its fresh namespace has not been executed. Smoke and pilot data would not be pooled into final inference.


In [4]:
print('First blocked example IDs:', example['blocked_first_ids'])
print('First interleaved example IDs:', example['interleaved_first_ids'])
print('Full ID multisets were verified equal:', example['example_id_multisets_verified_equal'])
pair = example['pair_checks']
print('Initial hashes match:', len(set(pair['initial_state_hashes'].values())) == 1)
print('Updates per schedule:', pair['updates_each_schedule'])


First blocked example IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
First interleaved example IDs: [8272, 6951, 2432, 13517, 13508, 2999, 12585, 3716, 746, 4947, 11558, 11606]
Full ID multisets were verified equal: True
Initial hashes match: True
Updates per schedule: 750


## Accuracy and learning curves

Action accuracy is the fraction of the seven action predictions that are correct. Complete-sequence accuracy requires all seven to be correct. A model can get most shared positions right yet fail the differing action, so these metrics answer different questions.

The primary outcome averages **fresh-noisy final action accuracy equally over A/B/C**. Clean accuracy is reported separately. Development probes track all three bases at updates 0,25,…,750 and never modify weights. The speed threshold requires at least 90% noisy mean accuracy for three consecutive checkpoints; otherwise it is `not_reached`.

![Pilot fixed development probes, diagnostic only](../results/pilot/figures/learning_curves.png)


In [5]:
runs = pd.read_csv(analysis / 'run_level.csv')
summary = pd.read_csv(analysis / 'primary_summary.csv')
print(summary[['distance', 'paired_runs', 'blocked_noisy_accuracy_pct',
               'IP_noisy_accuracy_pct', 'mean_difference_pp']].round(2).to_string(index=False))
print('\nThreshold reached counts:')
print(pd.read_csv(analysis / 'threshold_reached_counts.csv').to_string(index=False))


 distance  paired_runs  blocked_noisy_accuracy_pct  IP_noisy_accuracy_pct  mean_difference_pp
        1            5                       90.48                  96.22                5.74
        2            5                       74.33                  82.26                7.92
        4            5                       62.40                  80.93               18.53
        7            5                       38.09                  72.79               34.70

Threshold reached counts:
 distance    schedule  reached_count
        1     blocked              5
        1 interleaved              5
        2     blocked              0
        2 interleaved              0
        4     blocked              0
        4 interleaved              1
        7     blocked              0
        7 interleaved              0


## Paired differences and bootstrap confidence intervals

For each run and k, subtract blocked accuracy from interleaved accuracy. Multiplying by 100 expresses the result in **percentage points**. Positive means IP was higher in that run.

The independent unit is the simulation run. A noisy copy or a position shares the same model and training history, so it cannot be treated as another independent subject.

The proposed final analysis would resample 30 simulation IDs with replacement 10,000 times, then average their paired differences. The 2.5th and 97.5th percentiles form a percentile 95% confidence interval. The slope analysis resamples whole IDs with **all four distances together**. This uncertainty concerns the artificial model/generator; it does not cover biological uncertainty. The five-ID pilot is not a substitute for that final analysis.


In [6]:
paired = pd.read_csv(analysis / 'paired_differences.csv')
print(paired.head(8).round(3).to_string(index=False))
matrix = paired.pivot(index='run_id', columns='distance', values='IP_minus_blocked_pp')
print('Pilot matrix: rows are independent IDs, columns are distances:', matrix.shape)
print('Future bootstrap samples complete rows, retaining all four columns together.')
print('Pilot estimates above are diagnostic; no final conclusions are drawn here.')


 run_id  distance  blocked  interleaved  IP_minus_blocked_pp
      0         1    0.905        0.940                3.571
      0         2    0.621        0.844               22.286
      0         4    0.667        0.682                1.571
      0         7    0.333        0.564               23.048
      1         1    0.905        0.999                9.429
      1         2    0.810        0.857                4.762
      1         4    0.635        0.989               35.333
      1         7    0.476        0.667               19.095
Pilot matrix: rows are independent IDs, columns are distances: (5, 4)
Future bootstrap samples complete rows, retaining all four columns together.
Pilot estimates above are diagnostic; no final conclusions are drawn here.


![Pilot categorical differences, diagnostic only](../results/pilot/figures/paired_differences.png)

Inspect all four values rather than relying only on a straight-line slope. A slope can conceal a nonlinear pattern. No p-values are required for this project.


## Learning D and forgetting

After main practice, both trained models are cloned and given the same 2000 examples of one unused D pattern, in the same order (100 updates). The old A/B/C probes are evaluated before D and after every update.

**Forgetting = old final accuracy before D − old final accuracy after D.** The identical held-out final set is used twice. Positive means an accuracy drop; negative means improvement. D distance is recorded but not manipulated, making this a secondary experiment. A model that starts near a floor has less accuracy available to lose, so raw forgetting differences need baseline context.

![Pilot A/B/C during D, diagnostic only](../results/pilot/figures/D_forgetting.png)


In [7]:
print('D distances to A/B/C for this example:', manifest['D_distances_to_ABC'])
secondary = pd.read_csv(analysis / 'secondary_summary.csv')
print(secondary[['distance', 'schedule', 'forgetting_clean', 'forgetting_noisy']]
      .round(3).to_string(index=False))


D distances to A/B/C for this example: [5, 4, 4]
 distance    schedule  forgetting_clean  forgetting_noisy
        1     blocked             0.429             0.430
        1 interleaved             0.448             0.446
        2     blocked             0.362             0.366
        2 interleaved             0.438             0.433
        4     blocked             0.257             0.271
        4 interleaved             0.419             0.417
        7     blocked             0.029             0.038
        7 interleaved             0.314             0.313


## Shared positions, unseen bases, and limits

Shared and differing positions are scored separately. For k=7 there are **zero shared positions**: shared accuracy is NaN, and count is zero. It is not a zero-percent accuracy result.

Generalization here means decoding **100 previously unseen complete cues**, not acquiring a new physical skill. Clean and noisy generalization are separate. Action frequencies and repeated-action patterns are recorded but not matched across k. The task uses ordinal cue values 1–4, only seven hidden units, a fixed training budget, and no offline consolidation process.

The pilot diagnostics and limitations are in `../reports/results.md` and `../reports/plain_english.md`. Use `../reports/professor_questions.md` to prepare a discussion about what the model can and cannot establish. This notebook does not report results for the unexecuted proposed confirmation.


In [8]:
positions = pd.read_csv(analysis / 'position_summary.csv')
print(positions[(positions['distance'] == 7) & (positions['group'] == 'shared')]
      [['schedule', 'split', 'action_accuracy', 'action_count']].to_string(index=False))
print('\nClean and noisy unseen-base accuracies:')
print(secondary[['distance', 'schedule', 'generalization_clean_accuracy',
                 'generalization_noisy_accuracy']].round(3).to_string(index=False))


   schedule                split  action_accuracy  action_count
    blocked          final_clean              NaN           0.0
    blocked          final_noisy              NaN           0.0
    blocked generalization_clean              NaN           0.0
    blocked generalization_noisy              NaN           0.0
interleaved          final_clean              NaN           0.0
interleaved          final_noisy              NaN           0.0
interleaved generalization_clean              NaN           0.0
interleaved generalization_noisy              NaN           0.0

Clean and noisy unseen-base accuracies:
 distance    schedule  generalization_clean_accuracy  generalization_noisy_accuracy
        1     blocked                          0.255                          0.255
        1 interleaved                          0.274                          0.273
        2     blocked                          0.255                          0.256
        2 interleaved                          